# 107 — SmolVLA stock-U20 gate calibration, indices 10–14

Collects two exact-matched arms on all 200 standard-LIBERO identities at state indices 10–14: stock with measurement-only P&P, and always-on stock/refine blend projected at `s=0.3, K=3`. Both execute 10 actions at a time. The final cell derives fixed U20 quantiles, outcome flips, and a first-boundary threshold sweep; indices 0–9 remain untouched for routing evaluation. This uses a fresh persistence namespace, so the partial s=0.5 calibration cannot be resumed or mixed in. Videos and large trajectory/training artifacts are off.

In [ ]:
# Install NVIDIA EGL before importing MuJoCo.
import re, subprocess
driver = subprocess.check_output(
    ['nvidia-smi','--query-gpu=driver_version','--format=csv,noheader'],
    text=True).splitlines()[0].strip()
package = f"libnvidia-gl-{re.match(r'\d+', driver).group()}"
if subprocess.run('ldconfig -p | grep -q libEGL_nvidia', shell=True).returncode:
    print('Installing NVIDIA EGL package:', package)
    subprocess.run(['apt-get','update','-qq'], check=True)
    subprocess.run(['apt-get','install','-y','--no-install-recommends',package], check=True)
subprocess.run('ldconfig -p | grep libEGL_nvidia', shell=True, check=True)

In [ ]:
EXTRAS = 'sim'
SETUP_ENV = True
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

In [ ]:
from pnp.smolvla_u20_gate_calibration import (
    SMOLVLA_U20_GATE_CALIBRATION_EXPERIMENT,
    run_smolvla_u20_gate_calibration_worker)

ROLLOUT_BATCH_SIZE = 8
print({'experiment': SMOLVLA_U20_GATE_CALIBRATION_EXPERIMENT,
       'calibration_indices': list(range(10, 15)),
       'evaluation_indices_reserved': list(range(10)),
       'identities': 200,
       'new_rollouts': 400,
       'arms': ['stock with measurement-only P&P',
                'always-on blend: stock/refine 50/50 -> s=0.3 K=3 projection'],
       'refine': False,
       'pnp_steps': [1, 2, 3], 'pnp_k_by_step': [3, 1, 1],
       'gate_signal': 'pair-weighted U20 at every decision boundary',
       'planned_gate_quantile': 0.75,
       'projection_s': 0.3, 'projection_step': 7, 'projection_k': 3,
       'integration_steps': 10, 'n_action_steps': 10,
       'generated_chunk_size': 50,
       'progress_every_identities': 10, 'video': 'off'})
run_smolvla_u20_gate_calibration_worker(
    rollout_batch_size=ROLLOUT_BATCH_SIZE)

In [ ]:
# Run after collection completes. Raw artifacts remain persisted remotely; this JSON is small.
from pnp.smolvla_u20_gate_calibration import summarize_smolvla_u20_gate_calibration

THRESHOLD_JSON = '/content/smolvla_stock_u20_gate_idx10_14_s03.json'
threshold_report = summarize_smolvla_u20_gate_calibration(
    output_path=THRESHOLD_JSON, require_complete=True)
print('Use the quantile tables and first-boundary proxy sweep to freeze the indices-0:9 gate.')